In [2]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

pd.set_option('display.max_colwidth', 100)

In [4]:
df = pd.read_csv("../../Data/Final_Dataset.csv")
print(df.shape)
df.head(2)

(1473649, 12)


,id,title,overview,genres,keywords,cast,director,poster_path,release_date,vote_average,runtime,tagline
0,2,Ariel,A Finnish man goes to the city to find a job after the mine where he worked is closed and his fa...,"Comedy, Drama, Romance, Crime","prison|underdog|helsinki, finland|factory worker|prisoner|falling in love","Turo Pajala, Susanna Haavisto, Matti Pellonpää, Eetu Hilkamo, Erkki Pajala, Matti Jaaranen, Hann...",Aki Kaurismäki,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,1988-10-21,7.118,73.0,NaN
1,3,Shadows in Paradise,"Nikander, a rubbish collector and would-be entrepreneur, finds his plans for success dashed when...","Comedy, Drama, Romance","helsinki, finland|salesclerk|garbage","Matti Pellonpää, Kati Outinen, Sakari Kuosmanen, Esko Nikkari, Kylli Köngäs, Pekka Laiho, Jukka-...",Aki Kaurismäki,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,1986-10-17,7.300,74.0,NaN


In [5]:
def clean_text(text):
    """
    Cleans a single string:
    - Convert to lowercase
    - Remove punctuation
    - Remove stopwords
    - Strip extra spaces
    """
    if pd.isna(text):
        text = ""
    text = str(text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_column(col, sep=None):
    """
    Cleans an entire column.
    sep: if the column has a separator (e.g. '|' for genres/keywords,
         ',' for cast), replace it with a space before cleaning so
         individual tokens aren't glued together.
    """
    col = col.fillna("")
    if sep:
        col = col.apply(lambda x: str(x).replace(sep, " "))
    return col.apply(clean_text)

In [6]:
df['overview_clean']  = clean_column(df['overview'])
df['genres_clean']    = clean_column(df['genres'], sep='|')
df['keywords_clean']  = clean_column(df['keywords'], sep='|')
df['cast_clean']      = clean_column(df['cast'], sep=',')
df['director_clean']  = clean_column(df['director'])

df[['overview_clean', 'genres_clean', 'keywords_clean',
    'cast_clean', 'director_clean']].head(3)

,overview_clean,genres_clean,keywords_clean,cast_clean,director_clean
0,finnish man goes city find job mine worked closed father commits suicide,comedy drama romance crime,prison underdog helsinki finland factory worker prisoner falling love,turo pajala susanna haavisto matti pellonpää eetu hilkamo erkki pajala matti jaaranen hannu viho...,aki kaurismäki
1,nikander rubbish collector wouldbe entrepreneur finds plans success dashed business associate di...,comedy drama romance,helsinki finland salesclerk garbage,matti pellonpää kati outinen sakari kuosmanen esko nikkari kylli köngäs pekka laiho jukkapekka p...,aki kaurismäki
2,ted bellhops first night joband hotels unusual guests place outrageous predicaments seems evenin...,comedy,hotel new years eve witch bet sperm hotel room anthology los angeles california hoodlum multiple...,sammi davis salma hayek pinault lawrence bender madonna jennifer beals alicia witt lili taylor p...,quentin tarantino robert rodriguez alexandre rockwell allison anders


In [7]:
weights = {
    'cast': 3,       # who's in it -> strong similarity signal
    'director': 3,   # directing style -> strong similarity signal
    'genres': 2,      # broad category signal
    'keywords': 2,    # specific plot/theme tags
    'overview': 1     # free text, noisiest -> lowest weight
}

weights

{'cast': 3, 'director': 3, 'genres': 2, 'keywords': 2, 'overview': 1}

In [8]:
def weighted_join(row):
    parts = []
    parts += [row['genres_clean']]   * weights['genres']
    parts += [row['keywords_clean']] * weights['keywords']
    parts += [row['cast_clean']]     * weights['cast']
    parts += [row['director_clean']] * weights['director']
    parts += [row['overview_clean']] * weights['overview']

    # Filter out empty strings before joining
    return ' '.join(p for p in parts if p)

df['Equivalent'] = df.apply(weighted_join, axis=1)

df[['title', 'Equivalent']].head(3)

,title,Equivalent
0,Ariel,comedy drama romance crime comedy drama romance crime prison underdog helsinki finland factory w...
1,Shadows in Paradise,comedy drama romance comedy drama romance helsinki finland salesclerk garbage helsinki finland s...
2,Four Rooms,comedy comedy hotel new years eve witch bet sperm hotel room anthology los angeles california ho...


In [9]:
df.drop(columns=['overview_clean', 'genres_clean', 'keywords_clean',
                  'cast_clean', 'director_clean'], inplace=True)

df.head(2)

,id,title,overview,genres,keywords,cast,director,poster_path,release_date,vote_average,runtime,tagline,Equivalent
0,2,Ariel,A Finnish man goes to the city to find a job after the mine where he worked is closed and his fa...,"Comedy, Drama, Romance, Crime","prison|underdog|helsinki, finland|factory worker|prisoner|falling in love","Turo Pajala, Susanna Haavisto, Matti Pellonpää, Eetu Hilkamo, Erkki Pajala, Matti Jaaranen, Hann...",Aki Kaurismäki,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,1988-10-21,7.118,73.0,NaN,comedy drama romance crime comedy drama romance crime prison underdog helsinki finland factory w...
1,3,Shadows in Paradise,"Nikander, a rubbish collector and would-be entrepreneur, finds his plans for success dashed when...","Comedy, Drama, Romance","helsinki, finland|salesclerk|garbage","Matti Pellonpää, Kati Outinen, Sakari Kuosmanen, Esko Nikkari, Kylli Köngäs, Pekka Laiho, Jukka-...",Aki Kaurismäki,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,1986-10-17,7.300,74.0,NaN,comedy drama romance comedy drama romance helsinki finland salesclerk garbage helsinki finland s...


In [10]:
print("Any nulls in Equivalent?:", df['Equivalent'].isna().sum())
print("Any empty strings in Equivalent?:", (df['Equivalent'] == "").sum())
print("\nSample Equivalent value:\n")
print(df['Equivalent'].iloc[0])

Any nulls in Equivalent?: 0
Any empty strings in Equivalent?: 72375

Sample Equivalent value:

comedy drama romance crime comedy drama romance crime prison underdog helsinki finland factory worker prisoner falling love prison underdog helsinki finland factory worker prisoner falling love turo pajala susanna haavisto matti pellonpää eetu hilkamo erkki pajala matti jaaranen hannu viholainen jorma markkula tarja keinänen eino kuusela kauko laalo jyrki olsonen esko nikkari marja packalén mikko remes tomi salmela reijo marin heikki salomaa veikko uusimäki esko salminen hannu kivisalo pekka wilen heikki anttila pentti auer timo harakka kari helaseppä juuso hirvikangas hanna jokinen sakari kuosmanen sami lanki jouko lumme mikko lyytikäinen timo markko merja pulkkinen markku rantala sirkka rautiainen jaakko talaskivi timo toikka olli varja turo pajala susanna haavisto matti pellonpää eetu hilkamo erkki pajala matti jaaranen hannu viholainen jorma markkula tarja keinänen eino kuusela kauko laal

In [11]:
df2=df.drop(columns=['overview', 'genres', 'keywords',
                  'cast', 'director'], inplace=False)

df2.head(2)

,id,title,poster_path,release_date,vote_average,runtime,tagline,Equivalent
0,2,Ariel,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,1988-10-21,7.118,73.0,NaN,comedy drama romance crime comedy drama romance crime prison underdog helsinki finland factory w...
1,3,Shadows in Paradise,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,1986-10-17,7.300,74.0,NaN,comedy drama romance comedy drama romance helsinki finland salesclerk garbage helsinki finland s...


In [14]:
df2["Equivalent"].isnull().sum()

np.int64(0)

In [15]:
df2["Equivalent"].str.len().describe()

count    1.473649e+06
mean     4.561456e+02
std      4.836427e+02
min      0.000000e+00
25%      1.540000e+02
50%      3.280000e+02
75%      6.000000e+02
max      2.225100e+04
Name: Equivalent, dtype: float64

In [16]:
before_count = len(df2)

# Filter out rows where Equivalent is empty string (length 0)
df2 = df2[df2["Equivalent"].str.len() > 0].reset_index(drop=True)

after_count = len(df2)
removed_count = before_count - after_count

print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")
print(f"Rows removed: {removed_count}")

Rows before: 1473649
Rows after: 1401274
Rows removed: 72375


In [17]:
print("Min length now:", df2["Equivalent"].str.len().min())
df2["Equivalent"].str.len().describe()

Min length now: 1


count    1.401274e+06
mean     4.797052e+02
std      4.844480e+02
min      1.000000e+00
25%      1.780000e+02
50%      3.480000e+02
75%      6.200000e+02
max      2.225100e+04
Name: Equivalent, dtype: float64

In [19]:
df2.shape

(1401274, 8)

In [20]:
df2.to_csv("../../Data/Final_Dataset_with_Equivalent.csv", index=False)
print("Saved successfully.")

Saved successfully.
